# City Conditions ETL Pipeline
### Daily weather + air quality analytics with next-day forecasting — Toronto, Canada

This notebook documents the full pipeline behind the [City Conditions ETL Pipeline](https://github.com/Data-Netrunner/City-Conditions-ETL-Pipeline).

Every morning, GitHub Actions:

1. **Extracts** hourly weather and air quality from Open-Meteo (no API key)
2. **Transforms** it — parsing, validating, deduplicating
3. **Loads** it into a DuckDB warehouse via idempotent upserts
4. **Appends** observed daily aggregates to a durable, version-controlled history file
5. **Forecasts** the next day's average temperature and scores every past forecast walk-forward
6. **Publishes** KPIs, charts, and a rewritten README

---

> **This notebook is generated, not hand-written.** Every code cell below is read directly from the repository source by `tools/build_notebook.py`. Regenerate it with:
>
> ```bash
> python tools/build_notebook.py
> ```
>
> Doing it this way means the notebook cannot drift out of sync with the code it describes — the previous hand-maintained version had done exactly that.


## Dependencies

The pipeline needs five libraries. In Colab, only `duckdb` is missing by default.


In [ ]:
!pip install -q duckdb pandas requests matplotlib scikit-learn

---
## 1. Configuration

`etl/config.py`

All location settings and file paths live in one place. To point this pipeline at a different city, this is the only file that changes.


In [ ]:
# etl/config.py

CITY     = "Toronto"
LAT      = 43.6532
LON      = -79.3832
TIMEZONE = "America/Toronto"

RAW_WEATHER_CSV = "data/raw/weather_hourly.csv"

---
## 2. Extract — Weather

`etl/extract_weather.py`

Pulls hourly weather from the Open-Meteo Forecast API. No API key required.

**Note `forecast_days=0`.** Open-Meteo returns past *and* future days in a single payload, defaulting to 7 forecast days. Loading those future rows into the observations table would store model output as measurement — and would hand a next-day forecasting model the answer it is supposed to predict. That is target leakage, and this parameter is what prevents it.


In [ ]:
# etl/extract_weather.py

import requests
import pandas as pd


def fetch_weather_hourly(
    lat: float,
    lon: float,
    timezone: str,
    past_days: int = 7,
    forecast_days: int = 0,
) -> pd.DataFrame:
    """
    Pull hourly weather from Open-Meteo.

    forecast_days defaults to 0 — this is deliberate and important.

    Open-Meteo's /forecast endpoint returns past days AND future days in one
    payload, and its default is 7 forecast days. Loading those future rows into
    fact_weather_hourly stores model output as if it were measurement: the
    README's "latest" KPI ends up being a forecast, and any next-day prediction
    model trained on the table is just learning to copy Open-Meteo.

    Pass forecast_days explicitly if you want the API's own forecast — but keep
    it in a separate table, not in the observations fact table.

    Returns: ts, temperature_c, precipitation_mm, windspeed_kmh
    """
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude":      lat,
        "longitude":     lon,
        "hourly":        "temperature_2m,precipitation,windspeed_10m",
        "past_days":     past_days,
        "forecast_days": forecast_days,
        "timezone":      timezone,
    }

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    j = r.json()

    hourly = j["hourly"]
    df = pd.DataFrame({
        "ts":               pd.to_datetime(hourly["time"]),
        "temperature_c":    hourly["temperature_2m"],
        "precipitation_mm": hourly["precipitation"],
        "windspeed_kmh":    hourly["windspeed_10m"],
    })

    return df

---
## 3. Extract — Air Quality

`etl/extract_openaq.py`

Pulls hourly PM2.5, PM10, NO2 and ozone from the Open-Meteo Air Quality API — the standard indicators of urban air quality.


In [ ]:
# etl/extract_openaq.py

import requests
import pandas as pd


def fetch_air_quality_hourly(lat: float, lon: float, timezone: str, past_days: int = 7) -> pd.DataFrame:
    """
    Pull past N days of hourly air quality from Open-Meteo Air Quality API.
    Returns: ts, pm25, pm10, no2, o3
    """
    url = "https://air-quality-api.open-meteo.com/v1/air-quality"
    params = {
        "latitude":  lat,
        "longitude": lon,
        "hourly":    "pm2_5,pm10,nitrogen_dioxide,ozone",
        "past_days": past_days,
        "timezone":  timezone,
    }

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    j = r.json()

    h = j["hourly"]
    df = pd.DataFrame({
        "ts":   pd.to_datetime(h["time"]),
        "pm25": h.get("pm2_5"),
        "pm10": h.get("pm10"),
        "no2":  h.get("nitrogen_dioxide"),
        "o3":   h.get("ozone"),
    })

    return df

---
## 4. Transform — Weather

`etl/transform_weather.py`

Parses timestamps, nulls physically impossible readings (temperatures outside −60 °C to 60 °C, negative rainfall or wind), deduplicates on timestamp, and sorts chronologically.


In [ ]:
# etl/transform_weather.py

import pandas as pd


def clean_weather(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["ts"] = pd.to_datetime(out["ts"], errors="coerce")
    out = out.dropna(subset=["ts"])

    # Sanity bounds
    out.loc[(out["temperature_c"] < -60) | (out["temperature_c"] > 60), "temperature_c"] = None
    out.loc[out["precipitation_mm"] < 0, "precipitation_mm"] = None
    out.loc[out["windspeed_kmh"]    < 0, "windspeed_kmh"]    = None

    out = out.drop_duplicates(subset=["ts"])
    out = out.sort_values("ts").reset_index(drop=True)

    return out

---
## 5. Transform — Air Quality

`etl/transform_air_quality.py`

Same treatment for air quality: negative concentrations are physically impossible and get nulled rather than silently averaged in.


In [ ]:
# etl/transform_air_quality.py

import pandas as pd


def clean_air_quality(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["ts"] = pd.to_datetime(out["ts"], errors="coerce")
    out = out.dropna(subset=["ts"])

    # Remove physically impossible negatives
    for col in ["pm25", "pm10", "no2", "o3"]:
        if col in out.columns:
            out.loc[out[col] < 0, col] = None

    out = out.drop_duplicates(subset=["ts"])
    out = out.sort_values("ts").reset_index(drop=True)

    return out

---
## 6. Warehouse Schema

`sql/schema.sql`

Three tables: `dim_location`, `fact_weather_hourly`, and `fact_air_quality_hourly`. Both fact tables use a composite primary key of `(location_id, ts)` — that is what makes the upsert idempotent, so running the pipeline twice in a day updates rows instead of duplicating them.

`CREATE TABLE IF NOT EXISTS` makes the schema safe to re-run on every pipeline start.


```sql
-- City Conditions warehouse schema
-- init_db() runs this on every pipeline start; CREATE TABLE IF NOT EXISTS makes it safe to re-run.

CREATE TABLE IF NOT EXISTS dim_location (
  location_id INTEGER PRIMARY KEY,
  city        VARCHAR,
  lat         DOUBLE,
  lon         DOUBLE,
  timezone    VARCHAR
);

CREATE TABLE IF NOT EXISTS fact_weather_hourly (
  location_id      INTEGER,
  ts               TIMESTAMP,
  temperature_c    DOUBLE,
  precipitation_mm DOUBLE,
  windspeed_kmh    DOUBLE,
  PRIMARY KEY (location_id, ts)
);

CREATE TABLE IF NOT EXISTS fact_air_quality_hourly (
  location_id INTEGER,
  ts          TIMESTAMP,
  pm25        DOUBLE,
  pm10        DOUBLE,
  no2         DOUBLE,
  o3          DOUBLE,
  PRIMARY KEY (location_id, ts)
);
```


---
## 7. Load — Weather

`etl/load_weather_duckdb.py`

`INSERT ... ON CONFLICT DO UPDATE` against DuckDB, a serverless analytical database that runs entirely from a single file.


In [ ]:
# etl/load_weather_duckdb.py

import duckdb
import pandas as pd


def init_db(db_path: str, schema_sql_path: str) -> None:
    """Create tables if they don't exist."""
    con = duckdb.connect(db_path)
    with open(schema_sql_path, "r", encoding="utf-8") as f:
        con.execute(f.read())
    con.close()


def upsert_location(db_path: str, location_id: int, city: str, lat: float, lon: float, timezone: str) -> None:
    con = duckdb.connect(db_path)
    con.execute("""
        INSERT INTO dim_location(location_id, city, lat, lon, timezone)
        VALUES (?, ?, ?, ?, ?)
        ON CONFLICT(location_id) DO UPDATE SET
          city     = excluded.city,
          lat      = excluded.lat,
          lon      = excluded.lon,
          timezone = excluded.timezone
    """, [location_id, city, lat, lon, timezone])
    con.close()


def upsert_weather(db_path: str, df_weather: pd.DataFrame, location_id: int = 1) -> None:
    con = duckdb.connect(db_path)
    df = df_weather.copy()
    df["location_id"] = location_id
    con.register("w", df)
    con.execute("""
        INSERT INTO fact_weather_hourly(location_id, ts, temperature_c, precipitation_mm, windspeed_kmh)
        SELECT location_id, ts, temperature_c, precipitation_mm, windspeed_kmh FROM w
        ON CONFLICT(location_id, ts) DO UPDATE SET
          temperature_c    = excluded.temperature_c,
          precipitation_mm = excluded.precipitation_mm,
          windspeed_kmh    = excluded.windspeed_kmh
    """)
    con.close()

---
## 8. Load — Air Quality

`etl/load_air_quality_duckdb.py`

The same upsert pattern for the air quality fact table.


In [ ]:
# etl/load_air_quality_duckdb.py

import duckdb
import pandas as pd


def upsert_air_quality(db_path: str, df_air: pd.DataFrame, location_id: int = 1) -> None:
    con = duckdb.connect(db_path)
    df = df_air.copy()
    df["location_id"] = location_id
    con.register("a", df)
    con.execute("""
        INSERT INTO fact_air_quality_hourly(location_id, ts, pm25, pm10, no2, o3)
        SELECT location_id, ts, pm25, pm10, no2, o3 FROM a
        ON CONFLICT(location_id, ts) DO UPDATE SET
          pm25 = excluded.pm25,
          pm10 = excluded.pm10,
          no2  = excluded.no2,
          o3   = excluded.o3
    """)
    con.close()

---
## 9. KPI Query

`sql/kpis_combined.sql`

Aggregates the hourly warehouse into daily KPIs, joining weather and air quality by day, most recent 30 days first.


```sql
-- Combined daily KPIs: Weather + Air Quality (last 30 days, location_id = 1)
WITH daily_weather AS (
  SELECT
    DATE(ts)             AS day,
    location_id,
    AVG(temperature_c)   AS avg_temp_c,
    MAX(temperature_c)   AS max_temp_c,
    SUM(precipitation_mm)AS total_precip_mm,
    AVG(windspeed_kmh)   AS avg_windspeed_kmh,
    MAX(windspeed_kmh)   AS max_windspeed_kmh
  FROM fact_weather_hourly
  GROUP BY 1, 2
),
daily_aq AS (
  SELECT
    DATE(ts)   AS day,
    location_id,
    AVG(pm25)  AS pm25_avg,
    MAX(pm25)  AS pm25_peak,
    AVG(pm10)  AS pm10_avg,
    MAX(pm10)  AS pm10_peak,
    AVG(no2)   AS no2_avg,
    MAX(no2)   AS no2_peak,
    AVG(o3)    AS o3_avg,
    MAX(o3)    AS o3_peak
  FROM fact_air_quality_hourly
  GROUP BY 1, 2
)
SELECT
  w.day,
  w.location_id,
  w.avg_temp_c,
  w.max_temp_c,
  w.total_precip_mm,
  w.avg_windspeed_kmh,
  w.max_windspeed_kmh,
  a.pm25_avg,
  a.pm25_peak,
  a.pm10_avg,
  a.pm10_peak,
  a.no2_avg,
  a.no2_peak,
  a.o3_avg,
  a.o3_peak
FROM daily_weather w
LEFT JOIN daily_aq a
  ON  w.day         = a.day
  AND w.location_id = a.location_id
ORDER BY w.day DESC
LIMIT 30;
```


---
## 10. Durable History

`etl/history_store.py`

**This module exists because of a defect.** The pipeline only ever pulls a rolling 7-day window, and the DuckDB warehouse was not committed between CI runs — so no matter how many hundreds of times the pipeline ran, the warehouse reset to whatever was last checked in. The run log showed 200+ successes against a warehouse holding 35 days with a four-month hole in it.

The fix is an append-only CSV of observed daily aggregates that *is* committed on every run. It diffs cleanly in git (unlike a binary `.duckdb`), and it demotes the warehouse to a rebuildable derived artifact rather than the only copy of the data.


In [ ]:
# etl/history_store.py

"""
Durable daily history.

WHY THIS FILE EXISTS
--------------------
The pipeline only ever pulls a rolling ~7-day window from Open-Meteo, and the
DuckDB warehouse was never committed back by the GitHub Action. That means the
warehouse silently reset to whatever was last checked in — so no matter how
many days the pipeline ran, it never accumulated more than a couple of weeks
of history. A forecasting model trained on that is training on nothing.

The fix is an append-only CSV of *observed* daily aggregates that IS committed
on every run. It is small, diffs cleanly in git (unlike a binary .duckdb), and
makes the warehouse a rebuildable derived artifact rather than the only copy
of the data.
"""

import os

import pandas as pd
import duckdb

HISTORY_CSV = "data/history/daily_observations.csv"

HISTORY_COLS = [
    "day", "location_id", "avg_temp_c", "max_temp_c",
    "total_precip_mm", "avg_windspeed_kmh", "hours_observed",
]


def _daily_from_warehouse(db_path: str, location_id: int = 1) -> pd.DataFrame:
    """
    Roll hourly rows up to one row per day.

    Only days strictly before today are taken. Today is still partial, and
    anything after today is an Open-Meteo *forecast* — storing those as
    observations is what would leak the answer into a next-day model.
    """
    con = duckdb.connect(db_path, read_only=True)
    df = con.execute(
        """
        SELECT
            DATE(ts)              AS day,
            location_id,
            AVG(temperature_c)    AS avg_temp_c,
            MAX(temperature_c)    AS max_temp_c,
            SUM(precipitation_mm) AS total_precip_mm,
            AVG(windspeed_kmh)    AS avg_windspeed_kmh,
            COUNT(*)              AS hours_observed
        FROM fact_weather_hourly
        WHERE location_id = ?
          AND DATE(ts) < CURRENT_DATE
        GROUP BY 1, 2
        ORDER BY 1
        """,
        [location_id],
    ).df()
    con.close()

    df["day"] = pd.to_datetime(df["day"]).dt.date.astype(str)

    # A day with only a few hours recorded would skew its own daily average.
    return df[df["hours_observed"] >= 20].reset_index(drop=True)


def append_daily_history(db_path: str, location_id: int = 1) -> pd.DataFrame:
    """
    Merge today's warehouse view into the committed history file.

    Upsert semantics on (day, location_id): a day already in the file gets
    refreshed (Open-Meteo revises recent observations), new days get added,
    and days that have aged out of the API's rolling window are preserved.
    """
    os.makedirs(os.path.dirname(HISTORY_CSV), exist_ok=True)

    fresh = _daily_from_warehouse(db_path, location_id)

    if os.path.exists(HISTORY_CSV):
        prior = pd.read_csv(HISTORY_CSV)
        prior["day"] = prior["day"].astype(str)
        merged = pd.concat([prior, fresh], ignore_index=True)
        # keep="last" -> the freshly pulled version of a day wins
        merged = merged.drop_duplicates(subset=["day", "location_id"], keep="last")
    else:
        merged = fresh

    merged = merged[HISTORY_COLS].sort_values("day").reset_index(drop=True)
    merged.to_csv(HISTORY_CSV, index=False)

    print(f"Daily history: {len(merged)} observed days on file "
          f"({merged['day'].min()} to {merged['day'].max()}).")
    return merged


def load_daily_history(location_id: int = 1) -> pd.DataFrame:
    """Read the accumulated history back for modelling."""
    if not os.path.exists(HISTORY_CSV):
        return pd.DataFrame(columns=HISTORY_COLS)

    df = pd.read_csv(HISTORY_CSV)
    df = df[df["location_id"] == location_id].copy()
    df["day"] = pd.to_datetime(df["day"])
    return df.sort_values("day").reset_index(drop=True)

---
## 11. Prediction — Next-Day Temperature

`etl/predict_temperature.py`

The forecasting model. Four design decisions worth calling out:

1. **Observed data only.** Trains off the history file, never the warehouse's forecast rows.
2. **RidgeCV, not a forest.** At this sample size a tree ensemble overfits, and trees cannot extrapolate at all.
3. **Persistence baseline on every metric.** A model that cannot beat "tomorrow will be like today" has no value, and the skill score is published whether it is positive or negative.
4. **Walk-forward validation.** A random train/test split on a time series leaks the future into the past.

It also carries two guards added after the pipeline published a 40.07 °C August forecast for Toronto: seasonal day-of-year features stay disabled until the history covers most of a year, and every prediction is bounded by a physical sanity check before publication.


In [ ]:
# etl/predict_temperature.py

"""
Next-day average temperature forecasting.

Design notes (these are the interview talking points):

1. OBSERVED DATA ONLY. Open-Meteo's forecast endpoint returns past days AND
   future days in the same payload. Training a "next-day" model on rows that
   are themselves Open-Meteo forecasts is target leakage: the model would just
   learn to copy the API. Everything here reads the observed-only history file.

2. RIDGE, NOT A FOREST. At n~50 daily rows a tree ensemble overfits, and trees
   cannot extrapolate at all. A regularised linear model is the honest choice
   at this sample size.

3. BASELINE OR IT DIDN'T HAPPEN. Every metric is reported next to persistence
   ("tomorrow = today"), a surprisingly strong weather baseline. A model that
   cannot beat persistence has no value, and saying so is the point.

4. WALK-FORWARD VALIDATION. A random train/test split on a time series leaks
   the future into the past. Scoring is expanding-window: to score day i, fit
   only on days < i.

5. TWO GUARDS AGAINST EXTRAPOLATION. Ridge's ability to extrapolate is a
   double-edged sword. Trained on 21 days spanning February and June and then
   asked about late August, the seasonal day-of-year features extrapolated to
   a 40 C forecast for Toronto. So:

     (a) The doy_sin/doy_cos seasonal pair is only used once the history
         actually covers most of a year. Below that the seasonal cycle is not
         identifiable and those features do more harm than good.

     (b) Every prediction is bounded by a physical sanity check before it is
         published, and any clamping is recorded rather than hidden.

   Both guards apply inside the backtest too, so the reported accuracy is the
   accuracy of what actually ships.
"""

import os
from datetime import date

import numpy as np
import pandas as pd

from etl.history_store import load_daily_history

from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

PREDICTIONS_CSV = "reports/predictions.csv"
METRICS_CSV     = "reports/prediction_metrics.csv"
BACKTEST_CSV    = "reports/prediction_backtest.csv"

# Always-available features, derived from days <= d to predict day d + 1.
BASE_FEATURES = [
    "temp_lag1", "temp_lag2", "temp_lag3",
    "temp_roll3", "temp_roll7",
    "temp_delta1",
    "precip_lag1", "wind_lag1",
]

# Seasonality. Only switched on once the history can actually support it.
SEASONAL_FEATURES = ["doy_sin", "doy_cos"]

# Everything build_features() produces — used for assembling the launch row.
FEATURE_COLS = BASE_FEATURES + SEASONAL_FEATURES

MIN_TRAIN_ROWS = 15    # below this there is not enough history to fit anything

# Seasonal features need real coverage of the annual cycle before they mean
# anything. Roughly: a year's worth of rows spanning most of a year.
SEASONAL_MIN_ROWS      = 120
SEASONAL_MIN_SPAN_DAYS = 300

# Toronto's largest day-over-day change in daily MEAN temperature sits well
# inside this. Anything beyond it is a model failure, not weather.
MAX_DAILY_SWING_C = 12.0


# ----------------------------------------------------------------------------
# 1. Load observed daily history
# ----------------------------------------------------------------------------
def load_daily_observed(location_id: int = 1) -> pd.DataFrame:
    """
    Read the accumulated daily observation history.

    Deliberately reads the committed history file rather than the warehouse:
    the warehouse only ever holds Open-Meteo's rolling window, so querying it
    directly caps the model at ~2 weeks of training data no matter how long
    the pipeline has been running.
    """
    return load_daily_history(location_id)


# ----------------------------------------------------------------------------
# 2. Feature engineering
# ----------------------------------------------------------------------------
def build_features(daily: pd.DataFrame) -> pd.DataFrame:
    """
    Turn the daily series into a supervised learning table.

    One row per day d, with features built only from information available on
    day d, and target = the observed average temperature on day d + 1.

    The history has calendar gaps. Reindexing onto a complete daily calendar
    first means lags spanning a gap come out as NaN and get dropped, rather
    than silently pairing February with June.
    """
    if daily.empty:
        return pd.DataFrame()

    full_range = pd.date_range(daily["day"].min(), daily["day"].max(), freq="D")
    d = daily.set_index("day").reindex(full_range)
    d.index.name = "day"

    # --- lagged temperature: what we knew as of day d ---
    d["temp_lag1"] = d["avg_temp_c"].shift(0)   # today's observed mean
    d["temp_lag2"] = d["avg_temp_c"].shift(1)
    d["temp_lag3"] = d["avg_temp_c"].shift(2)

    # --- smoothed levels: rolling means damp single-day noise ---
    d["temp_roll3"] = d["avg_temp_c"].rolling(3).mean()
    d["temp_roll7"] = d["avg_temp_c"].rolling(7).mean()

    # --- momentum: is it warming or cooling? ---
    d["temp_delta1"] = d["avg_temp_c"] - d["avg_temp_c"].shift(1)

    # --- other same-day conditions ---
    d["precip_lag1"] = d["total_precip_mm"]
    d["wind_lag1"]   = d["avg_windspeed_kmh"]

    # --- seasonality: day-of-year as a circle, so Dec 31 sits next to Jan 1 ---
    doy = d.index.dayofyear
    d["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
    d["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)

    # --- target: tomorrow's observed average temperature ---
    d["target_temp_c"] = d["avg_temp_c"].shift(-1)
    d["target_day"]    = d.index + pd.Timedelta(days=1)

    return d.reset_index()


def training_table(feat: pd.DataFrame) -> pd.DataFrame:
    """Rows usable for fitting: every feature and the target must be present."""
    if feat.empty:
        return feat
    return feat.dropna(subset=FEATURE_COLS + ["target_temp_c"]).reset_index(drop=True)


def select_features(train_slice: pd.DataFrame) -> list:
    """
    Decide whether the seasonal pair has earned its place.

    With only a few weeks of gappy history, doy_sin/doy_cos are not seasonality
    — they are two arbitrary numbers that happen to separate the February rows
    from the June rows, and the model leans on them hard. Asked about a date
    outside that range, it extrapolates the sine wave into nonsense (this is
    exactly how a 40 C August forecast for Toronto got published).

    Computed from the slice the model is actually fitted on, so the backtest
    makes the same decision the production run would have made on that date.
    """
    if len(train_slice) < SEASONAL_MIN_ROWS:
        return list(BASE_FEATURES)

    span_days = (train_slice["day"].max() - train_slice["day"].min()).days
    if span_days < SEASONAL_MIN_SPAN_DAYS:
        return list(BASE_FEATURES)

    return list(BASE_FEATURES) + list(SEASONAL_FEATURES)


def apply_sanity_bounds(pred: float, baseline: float,
                        train_slice: pd.DataFrame):
    """
    Bound the prediction by physical plausibility before publishing it.

    Two constraints, whichever is tighter:
      - within MAX_DAILY_SWING_C of today's observed mean
      - within 5 C of the range of temperatures ever observed

    Returns (bounded_prediction, was_clamped). A clamp firing is a signal the
    model is unhealthy, so it gets recorded in predictions.csv rather than
    quietly swallowed.
    """
    lo = max(baseline - MAX_DAILY_SWING_C, train_slice["target_temp_c"].min() - 5.0)
    hi = min(baseline + MAX_DAILY_SWING_C, train_slice["target_temp_c"].max() + 5.0)

    if lo > hi:  # degenerate history — fall back to the swing bound alone
        lo, hi = baseline - MAX_DAILY_SWING_C, baseline + MAX_DAILY_SWING_C

    bounded = float(np.clip(pred, lo, hi))
    return bounded, bool(abs(bounded - pred) > 1e-9)


def _new_model():
    """
    RidgeCV picks its own regularisation strength by cross-validation, so there
    is no hand-tuned magic number. Scaling first matters: ridge penalises large
    coefficients, and the raw features are on wildly different scales.
    """
    return make_pipeline(
        StandardScaler(),
        RidgeCV(alphas=np.logspace(-3, 3, 25)),
    )


# ----------------------------------------------------------------------------
# 3. Walk-forward backtest
# ----------------------------------------------------------------------------
def backtest(train: pd.DataFrame, min_train: int = MIN_TRAIN_ROWS) -> pd.DataFrame:
    """
    Expanding-window validation. For each day i past the warm-up, fit on days
    0..i-1 only and predict day i.

    Feature selection and sanity bounds are applied per fold, so these numbers
    describe the system that actually ships — not an idealised version of it.
    """
    rows = []
    if len(train) <= min_train:
        return pd.DataFrame(rows)

    for i in range(min_train, len(train)):
        past = train.iloc[:i]
        cols = select_features(past)

        model = _new_model()
        model.fit(past[cols], past["target_temp_c"])

        raw      = float(model.predict(train.iloc[i:i + 1][cols])[0])
        baseline = float(train.loc[i, "temp_lag1"])
        pred, clamped = apply_sanity_bounds(raw, baseline, past)

        rows.append({
            "target_day":       train.loc[i, "target_day"],
            "actual_temp_c":    float(train.loc[i, "target_temp_c"]),
            "predicted_temp_c": pred,
            "raw_model_temp_c": raw,
            "clamped":          clamped,
            # Persistence baseline: "tomorrow will be like today".
            "baseline_temp_c":  baseline,
            "n_features":       len(cols),
        })

    out = pd.DataFrame(rows)
    out["model_abs_err"]    = (out["predicted_temp_c"] - out["actual_temp_c"]).abs()
    out["baseline_abs_err"] = (out["baseline_temp_c"]  - out["actual_temp_c"]).abs()
    return out


def summarise(bt: pd.DataFrame) -> dict:
    """MAE / RMSE for model and baseline, plus forecast skill score."""
    if bt.empty:
        return {}

    def _rmse(err):
        return float(np.sqrt((err ** 2).mean()))

    model_mae  = float(bt["model_abs_err"].mean())
    base_mae   = float(bt["baseline_abs_err"].mean())
    model_rmse = _rmse(bt["predicted_temp_c"] - bt["actual_temp_c"])
    base_rmse  = _rmse(bt["baseline_temp_c"]  - bt["actual_temp_c"])

    # Skill: the share of the baseline's error the model removes. Positive =
    # better than persistence, negative = worse. Published either way.
    skill = (base_mae - model_mae) / base_mae if base_mae > 0 else float("nan")

    return {
        "n_scored_days":        int(len(bt)),
        "model_mae_c":          round(model_mae, 3),
        "model_rmse_c":         round(model_rmse, 3),
        "baseline_mae_c":       round(base_mae, 3),
        "baseline_rmse_c":      round(base_rmse, 3),
        "skill_vs_persistence": round(skill, 3),
        "n_clamped":            int(bt["clamped"].sum()),
    }


# ----------------------------------------------------------------------------
# 4. Predict tomorrow + persist outputs
# ----------------------------------------------------------------------------
def predict_next_day(location_id: int = 1) -> dict:
    """
    Full prediction step. Returns a dict the README writer consumes.
    Degrades gracefully: too little history skips the forecast rather than
    crashing the pipeline.
    """
    os.makedirs("reports", exist_ok=True)

    daily = load_daily_observed(location_id)
    feat  = build_features(daily)
    train = training_table(feat)

    result = {
        "trained_rows":     int(len(train)),
        "target_day":       None,
        "predicted_temp_c": None,
        "baseline_temp_c":  None,
        "seasonal_used":    False,
        "n_features":       0,
        "clamped":          False,
        "metrics":          {},
    }

    if len(train) < MIN_TRAIN_ROWS:
        print(f"Only {len(train)} usable training rows "
              f"(need {MIN_TRAIN_ROWS}) — prediction skipped this run.")
        return result

    # --- backtest on everything we have, then report it ---
    bt      = backtest(train)
    metrics = summarise(bt)
    result["metrics"] = metrics

    if not bt.empty:
        bt_out = bt.copy()
        bt_out["target_day"] = pd.to_datetime(bt_out["target_day"]).dt.date
        bt_out.to_csv(BACKTEST_CSV, index=False)

    # --- fit on ALL history, then forecast the next day ---
    cols = select_features(train)
    result["seasonal_used"] = "doy_sin" in cols
    result["n_features"]    = len(cols)

    model = _new_model()
    model.fit(train[cols], train["target_temp_c"])

    # Most recent day with a complete feature row is the launch point, even
    # though its target (tomorrow) hasn't happened yet.
    launch = feat.dropna(subset=FEATURE_COLS)
    launch = launch[launch["day"] == launch["day"].max()]

    raw           = float(model.predict(launch[cols])[0])
    baseline      = float(launch["temp_lag1"].iloc[0])
    pred, clamped = apply_sanity_bounds(raw, baseline, train)
    target_day    = pd.Timestamp(launch["day"].iloc[0]) + pd.Timedelta(days=1)

    result.update({
        "target_day":       target_day.date().isoformat(),
        "predicted_temp_c": round(pred, 2),
        "baseline_temp_c":  round(baseline, 2),
        "clamped":          clamped,
    })

    # --- append to the prediction log (one row per forecast ever made) ---
    row = pd.DataFrame([{
        "predicted_on":     date.today().isoformat(),
        "target_day":       result["target_day"],
        "predicted_temp_c": result["predicted_temp_c"],
        "raw_model_temp_c": round(raw, 2),
        "clamped":          clamped,
        "baseline_temp_c":  result["baseline_temp_c"],
        "trained_rows":     result["trained_rows"],
        "n_features":       len(cols),
        "seasonal_used":    result["seasonal_used"],
        "model":            "RidgeCV(standardised)",
    }])

    if os.path.exists(PREDICTIONS_CSV):
        prev = pd.read_csv(PREDICTIONS_CSV)
        # Re-running on the same day should overwrite, not duplicate.
        prev = prev[prev["predicted_on"] != row["predicted_on"].iloc[0]]
        row  = pd.concat([prev, row], ignore_index=True)
    row.to_csv(PREDICTIONS_CSV, index=False)

    # --- append metrics history so accuracy is trackable over time ---
    if metrics:
        mrow = pd.DataFrame([{"run_date": date.today().isoformat(), **metrics}])
        if os.path.exists(METRICS_CSV):
            prevm = pd.read_csv(METRICS_CSV)
            prevm = prevm[prevm["run_date"] != mrow["run_date"].iloc[0]]
            mrow  = pd.concat([prevm, mrow], ignore_index=True)
        mrow.to_csv(METRICS_CSV, index=False)

    print(f"Forecast for {result['target_day']}: {result['predicted_temp_c']} C "
          f"(persistence baseline {result['baseline_temp_c']} C)")
    print(f"  features: {len(cols)} "
          f"({'seasonal ON' if result['seasonal_used'] else 'seasonal OFF — not enough annual coverage'})")
    if clamped:
        print(f"  WARNING: raw model output {raw:.2f} C was outside physical "
              f"bounds and was clamped to {pred:.2f} C.")
    if metrics:
        print(f"  backtest over {metrics['n_scored_days']} days — "
              f"model MAE {metrics['model_mae_c']} C vs "
              f"persistence MAE {metrics['baseline_mae_c']} C "
              f"(skill {metrics['skill_vs_persistence']:+.1%})")

    return result

---
## 12. Reporting — Weather & Air Quality Charts

`etl/make_charts_combined.py`

Three 30-day time-series charts. `matplotlib.use("Agg")` is set explicitly because GitHub Actions runners have no display.


In [ ]:
# etl/make_charts_combined.py

import os

import matplotlib
matplotlib.use("Agg")  # Required for headless environments (GitHub Actions has no display)
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd


def _style_time_axis() -> None:
    ax = plt.gca()
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=6, maxticks=10))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    plt.xticks(rotation=45, ha="right")
    ax.grid(True, which="major", axis="both", linestyle="--", linewidth=0.5)


def make_charts(kpis_csv: str, charts_dir: str = "reports/charts") -> None:
    os.makedirs(charts_dir, exist_ok=True)

    df = pd.read_csv(kpis_csv)
    if df.empty:
        print("No KPI data — charts skipped.")
        return

    df["day"] = pd.to_datetime(df["day"])
    df = df.sort_values("day")

    # 1) Average Temperature (30d)
    plt.figure()
    plt.plot(df["day"], df["avg_temp_c"], label="Avg Temp")
    plt.title("Average Temperature (Last 30 Days)")
    plt.xlabel("Date")
    plt.ylabel("Temperature (°C)")
    _style_time_axis()
    plt.legend(loc="best")
    plt.tight_layout()
    plt.savefig(os.path.join(charts_dir, "avg_temp_30d.png"))
    plt.close()

    # 2) Total Precipitation (30d)
    plt.figure()
    plt.plot(df["day"], df["total_precip_mm"], label="Total Precip")
    plt.title("Total Precipitation per Day (Last 30 Days)")
    plt.xlabel("Date")
    plt.ylabel("Precipitation (mm)")
    _style_time_axis()
    plt.legend(loc="best")
    plt.tight_layout()
    plt.savefig(os.path.join(charts_dir, "precip_30d.png"))
    plt.close()

    # 3) PM2.5 Average (30d)
    if "pm25_avg" in df.columns:
        plt.figure()
        plt.plot(df["day"], df["pm25_avg"], label="PM2.5 Avg")
        plt.title("PM2.5 Average (Last 30 Days)")
        plt.xlabel("Date")
        plt.ylabel("PM2.5 (µg/m³)")
        _style_time_axis()
        plt.legend(loc="best")
        plt.tight_layout()
        plt.savefig(os.path.join(charts_dir, "pm25_avg_30d.png"))
        plt.close()

---
## 13. Reporting — Forecast Charts

`etl/make_prediction_chart.py`

Predicted vs actual vs baseline, and absolute error per day. The second chart is the one that shows *where* the model struggles rather than just how much.


In [ ]:
# etl/make_prediction_chart.py

"""Charts for the prediction section: predicted vs actual, and error over time."""

import os

import matplotlib
matplotlib.use("Agg")  # Required for headless environments (GitHub Actions has no display)
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

BACKTEST_CSV = "reports/prediction_backtest.csv"


def _style_time_axis() -> None:
    ax = plt.gca()
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=5, maxticks=10))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    plt.xticks(rotation=45, ha="right")
    ax.grid(True, which="major", axis="both", linestyle="--", linewidth=0.5)


def make_prediction_charts(charts_dir: str = "reports/charts", last_n: int = 30) -> None:
    """
    Two charts:
      1. predicted vs actual vs persistence baseline — does the line track?
      2. absolute error per day, model vs baseline — where does it go wrong?
    """
    if not os.path.exists(BACKTEST_CSV):
        print("No backtest file yet — prediction charts skipped.")
        return

    os.makedirs(charts_dir, exist_ok=True)

    df = pd.read_csv(BACKTEST_CSV)
    if df.empty:
        print("Backtest file is empty — prediction charts skipped.")
        return

    df["target_day"] = pd.to_datetime(df["target_day"])
    df = df.sort_values("target_day").tail(last_n)

    # --- 1) Predicted vs actual ------------------------------------------
    plt.figure()
    plt.plot(df["target_day"], df["actual_temp_c"],
             label="Actual", linewidth=2)
    plt.plot(df["target_day"], df["predicted_temp_c"],
             label="Model forecast", linestyle="--", marker="o", markersize=3)
    plt.plot(df["target_day"], df["baseline_temp_c"],
             label="Persistence baseline", linestyle=":", alpha=0.7)
    plt.title(f"Next-Day Temperature Forecast vs Actual (Last {len(df)} Scored Days)")
    plt.xlabel("Date")
    plt.ylabel("Average Temperature (°C)")
    _style_time_axis()
    plt.legend(loc="best")
    plt.tight_layout()
    plt.savefig(os.path.join(charts_dir, "forecast_vs_actual_30d.png"))
    plt.close()

    # --- 2) Absolute error per day ---------------------------------------
    plt.figure()
    plt.plot(df["target_day"], df["model_abs_err"],
             label="Model abs error", marker="o", markersize=3)
    plt.plot(df["target_day"], df["baseline_abs_err"],
             label="Baseline abs error", linestyle=":", alpha=0.8)
    plt.axhline(df["model_abs_err"].mean(), linestyle="--", linewidth=1,
                label=f"Model MAE = {df['model_abs_err'].mean():.2f} °C")
    plt.title("Forecast Absolute Error per Day")
    plt.xlabel("Date")
    plt.ylabel("Absolute error (°C)")
    _style_time_axis()
    plt.legend(loc="best")
    plt.tight_layout()
    plt.savefig(os.path.join(charts_dir, "forecast_error_30d.png"))
    plt.close()

    print("Prediction charts written.")

---
## 14. Reporting — README Forecast Block

`etl/readme_prediction_section.py`

Builds the forecast and accuracy markdown, including the disclosure lines about disabled seasonal features and any clamping applied.


In [ ]:
# etl/readme_prediction_section.py

"""
Builds the README's prediction block.

Kept in its own module so update_readme_weather.py only needs a three-line
change rather than a rewrite.
"""


def prediction_section_lines(prediction: dict | None) -> list[str]:
    """Return the markdown lines for the forecast + accuracy section."""
    if not prediction or prediction.get("predicted_temp_c") is None:
        return []

    m = prediction.get("metrics") or {}
    lines = []

    lines.append("## Next-Day Forecast (model output)\n\n")
    lines.append(
        f"- Forecast for **{prediction['target_day']}**: "
        f"**{prediction['predicted_temp_c']} °C** average temperature\n"
    )
    lines.append(
        f"- Persistence baseline (\"same as today\"): "
        f"{prediction['baseline_temp_c']} °C\n"
    )
    n_feat = prediction.get("n_features") or 0
    lines.append(
        f"- Model: RidgeCV on {n_feat} engineered features, "
        f"trained on {prediction['trained_rows']} observed days\n"
    )
    if not prediction.get("seasonal_used", False):
        lines.append(
            "- Seasonal (day-of-year) features are **disabled** until the history "
            "covers most of a year — with partial coverage they extrapolate badly\n"
        )
    if prediction.get("clamped", False):
        lines.append(
            "- This forecast was **clamped** to physically plausible bounds; "
            "the raw model output was out of range\n"
        )
    lines.append("\n")

    if m:
        skill = m.get("skill_vs_persistence")
        verdict = (
            "beating the baseline" if skill is not None and skill > 0
            else "not yet beating the baseline"
        )

        lines.append("### Accuracy (walk-forward backtest)\n\n")
        lines.append(
            "Every forecast below was made using only data available *before* "
            "the day it predicts — no future information leaks into training.\n\n"
        )
        lines.append("| Metric | Model | Persistence baseline |\n")
        lines.append("|---|---|---|\n")
        lines.append(f"| MAE (°C) | **{m['model_mae_c']}** | {m['baseline_mae_c']} |\n")
        lines.append(f"| RMSE (°C) | **{m['model_rmse_c']}** | {m['baseline_rmse_c']} |\n")
        lines.append(f"| Days scored | {m['n_scored_days']} | {m['n_scored_days']} |\n\n")

        if skill is not None:
            lines.append(
                f"**Skill score vs persistence: {skill:+.1%}** — currently {verdict}. "
                "Skill is the share of the baseline's error the model removes; "
                "it is published whether it is positive or negative.\n\n"
            )

        lines.append("![Forecast vs Actual](reports/charts/forecast_vs_actual_30d.png)\n\n")
        lines.append("![Forecast Error](reports/charts/forecast_error_30d.png)\n\n")

    lines.append("---\n\n")
    return lines

---
## 15. Reporting — README Rewrite

`etl/update_readme_weather.py`

Regenerates the README on every run so the published numbers can never be stale relative to the data.


In [ ]:
# etl/update_readme_weather.py

import math
import pandas as pd

from etl.readme_prediction_section import prediction_section_lines

def _fmt(x):
    try:
        if x is None or (isinstance(x, float) and math.isnan(x)):
            return "NA"
        return str(round(float(x), 2))
    except Exception:
        return "NA"

def update_readme(readme_path, kpis_csv_path, prediction=None):
    df = pd.read_csv(kpis_csv_path)
    latest = df.iloc[0].to_dict() if not df.empty else {}
    lines = []
    lines.append("# City Conditions ETL Pipeline\n\n")
    lines.append("A fully automated end-to-end ETL pipeline that pulls daily weather and air quality data for **Toronto, Canada**, loads it into a DuckDB analytics warehouse, and publishes KPI reports and charts updated every day via GitHub Actions.\n\n")
    lines.append("---\n\n")
    lines.append("## Data Sources\n\n")
    lines.append("All data is pulled from **[Open-Meteo](https://open-meteo.com/)** — a free, open-source weather and air quality API requiring no API key.\n\n")
    lines.append("| Source | API | Data Collected |\n")
    lines.append("|---|---|---|\n")
    lines.append("| Weather | [Open-Meteo Forecast API](https://api.open-meteo.com/v1/forecast) | Temperature (C), Precipitation (mm), Wind Speed (km/h) |\n")
    lines.append("| Air Quality | [Open-Meteo Air Quality API](https://air-quality-api.open-meteo.com/v1/air-quality) | PM2.5, PM10, NO2, Ozone |\n\n")
    lines.append("---\n\n")
    if latest:
        lines.append("## Latest KPI Snapshot\n\n")
        lines.append(f"- Date: **{latest.get('day')}**\n")
        lines.append(f"- Avg Temp (C): **{_fmt(latest.get('avg_temp_c'))}**\n")
        lines.append(f"- Max Temp (C): **{_fmt(latest.get('max_temp_c'))}**\n")
        lines.append(f"- Total Precip (mm): **{_fmt(latest.get('total_precip_mm'))}**\n")
        lines.append(f"- Avg Wind (km/h): **{_fmt(latest.get('avg_windspeed_kmh'))}**\n")
        lines.append(f"- Max Wind (km/h): **{_fmt(latest.get('max_windspeed_kmh'))}**\n")
        if "pm25_avg" in latest:
            lines.append(f"- PM2.5 Avg (ug/m3): **{_fmt(latest.get('pm25_avg'))}**\n")
            lines.append(f"- PM2.5 Peak (ug/m3): **{_fmt(latest.get('pm25_peak'))}**\n")
        lines.append("\n")
    lines.append("---\n\n")
    # --- model forecast + accuracy ---
    lines.extend(prediction_section_lines(prediction))

    lines.append("## Charts (auto-updated daily)\n\n")
    lines.append("### Weather\n\n")
    lines.append("- **Average Temperature (C):** daily mean temperature\n\n")
    lines.append("![Average Temperature (30d)](reports/charts/avg_temp_30d.png)\n\n")
    lines.append("- **Total Precipitation (mm):** total precipitation per day\n\n")
    lines.append("![Daily Precipitation (30d)](reports/charts/precip_30d.png)\n\n")
    lines.append("### Air Quality\n\n")
    lines.append("- **PM2.5 (ug/m3):** daily average fine particulate concentration (smaller = cleaner air)\n\n")
    lines.append("![PM2.5 Average (30d)](reports/charts/pm25_avg_30d.png)\n\n")
    lines.append("---\n\n")
    lines.append("## How It Works\n\n")
    lines.append("| Step | What happens |\n")
    lines.append("|---|---|\n")
    lines.append("| Extract | Pulls 7 days of hourly weather and air quality from Open-Meteo (observations only — `forecast_days=0`, so the API's own forecast never enters the observation tables) |\n")
    lines.append("| Transform | Cleans and validates: parses timestamps, nulls physically impossible values, deduplicates |\n")
    lines.append("| Load | Upserts into a DuckDB warehouse, keyed on (location_id, ts) so re-runs are idempotent |\n")
    lines.append("| History | Appends observed daily aggregates to `data/history/daily_observations.csv` — the durable record the model trains on |\n")
    lines.append("| Predict | Fits a RidgeCV model on lagged daily features, forecasts the next day, and scores every past forecast walk-forward |\n")
    lines.append("| Report | Generates KPI CSV, 30-day charts, forecast charts, and rewrites this README |\n")
    lines.append("| Log | Appends a row to reports/run_log.csv |\n\n")
    lines.append("---\n\n")
    lines.append("## Tech Stack\n\n")
    lines.append("| Technology | Purpose |\n")
    lines.append("|---|---|\n")
    lines.append("| Python | ETL scripting |\n")
    lines.append("| DuckDB | Analytics warehouse |\n")
    lines.append("| pandas | Data transformation |\n")
    lines.append("| requests | API extraction |\n")
    lines.append("| matplotlib | Chart generation |\n")
    lines.append("| scikit-learn | Forecasting model (RidgeCV) and validation |\n")
    lines.append("| pytest | Test suite |\n")
    lines.append("| GitHub Actions | Daily automation and CI |\n\n")
    lines.append("---\n\n")
    lines.append("## Outputs\n\n")
    lines.append("- `data/history/daily_observations.csv` — **the durable observation record.** Append-only, committed on every run, and what the forecasting model trains on\n")
    lines.append("- `warehouse/city_conditions.duckdb` — DuckDB warehouse, rebuilt from the API's rolling window each run (a derived artifact, not the system of record)\n")
    lines.append("- `reports/latest_kpis.csv` — most recent daily KPI snapshot\n")
    lines.append("- `reports/predictions.csv` — every forecast ever published, with the raw model output and whether sanity bounds were applied\n")
    lines.append("- `reports/prediction_backtest.csv` — per-day walk-forward scores: prediction, actual, and baseline\n")
    lines.append("- `reports/prediction_metrics.csv` — MAE / RMSE / skill history over time\n")
    lines.append("- `reports/run_log.csv` — log of every pipeline run\n")
    lines.append("- `reports/charts/*.png` — auto-generated 30-day charts\n\n")
    lines.append("---\n\n")
    lines.append("## Tests\n\n")
    lines.append("```bash\n")
    lines.append("pip install -r requirements-dev.txt\n")
    lines.append("pytest -q\n")
    lines.append("```\n\n")
    lines.append("The suite covers transform invariants, the no-leakage-across-calendar-gaps property of the feature builder, and regression tests for both extrapolation guards — including the literal 40 °C forecast this pipeline once published.\n\n")
    lines.append("---\n\n")
    lines.append("## How to Run Locally\n\n")
    lines.append("```bash\n")
    lines.append("git clone https://github.com/Data-Netrunner/City-Conditions-ETL-Pipeline.git\n")
    lines.append("cd City-Conditions-ETL-Pipeline\n")
    lines.append("pip install -r requirements.txt\n")
    lines.append("python etl/run_weather_pipeline.py\n")
    lines.append("```\n\n")
    lines.append("No API keys or paid services required.\n\n")
    lines.append("---\n\n")
    lines.append("*Built by Andre Felix - Data updated daily via GitHub Actions*\n")
    with open(readme_path, "w", encoding="utf-8") as f:
        f.writelines(lines)

---
## 16. Data Quality & Run Logging

`etl/data_quality.py`

Validation helpers that fail loudly before bad data reaches the warehouse, plus an append to `reports/run_log.csv` on every run — success or failure — giving a full history of pipeline health.


In [ ]:
# etl/data_quality.py

import os
from datetime import datetime, timezone

import pandas as pd

LOG_PATH = "reports/run_log.csv"


def assert_required_columns(df: pd.DataFrame, required: list[str], df_name: str) -> None:
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{df_name} missing required columns: {missing}")


def assert_not_empty(df: pd.DataFrame, df_name: str) -> None:
    if df is None or df.empty:
        raise ValueError(f"{df_name} is empty — nothing to load")


def append_run_log(
    weather_rows: int,
    aq_rows: int,
    kpi_rows: int,
    status: str,
    message: str = "",
) -> None:
    os.makedirs("reports", exist_ok=True)

    row = pd.DataFrame([{
        "run_ts_utc":    datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S"),
        "weather_rows":  weather_rows,
        "aq_rows":       aq_rows,
        "kpi_rows":      kpi_rows,
        "status":        status,
        "message":       message,
    }])

    if os.path.exists(LOG_PATH):
        existing = pd.read_csv(LOG_PATH)
        out = pd.concat([existing, row], ignore_index=True)
    else:
        out = row

    out.to_csv(LOG_PATH, index=False)

---
## 17. Main Pipeline Entry Point

`etl/run_weather_pipeline.py`

What GitHub Actions calls every morning. Orchestrates every step above, wrapped in try/except so a failure is logged to the run log before the error re-raises and turns the Actions run red.


In [ ]:
# etl/run_weather_pipeline.py

import os

import matplotlib
matplotlib.use("Agg")  # Must be set before any other matplotlib import (no display in GitHub Actions)

import duckdb

from etl.config import CITY, LAT, LON, TIMEZONE, RAW_WEATHER_CSV
from etl.extract_weather import fetch_weather_hourly
from etl.transform_weather import clean_weather
from etl.load_weather_duckdb import init_db, upsert_location, upsert_weather
from etl.extract_openaq import fetch_air_quality_hourly
from etl.transform_air_quality import clean_air_quality
from etl.load_air_quality_duckdb import upsert_air_quality
from etl.make_charts_combined import make_charts
from etl.update_readme_weather import update_readme
from etl.data_quality import assert_required_columns, assert_not_empty, append_run_log

# --- prediction section ---
from etl.history_store import append_daily_history
from etl.predict_temperature import predict_next_day
from etl.make_prediction_chart import make_prediction_charts

DB_PATH    = "warehouse/city_conditions.duckdb"
SCHEMA_SQL = "sql/schema.sql"
KPI_SQL    = "sql/kpis_combined.sql"
KPI_OUT    = "reports/latest_kpis.csv"


def main() -> None:
    weather_rows = 0
    aq_rows      = 0
    kpi_rows     = 0
    prediction   = None

    try:
        os.makedirs("data/raw",       exist_ok=True)
        os.makedirs("data/history",   exist_ok=True)
        os.makedirs("warehouse",      exist_ok=True)
        os.makedirs("reports",        exist_ok=True)
        os.makedirs("reports/charts", exist_ok=True)

        # 0) Schema + location seed
        init_db(DB_PATH, SCHEMA_SQL)
        upsert_location(DB_PATH, 1, CITY, LAT, LON, TIMEZONE)

        # 1) Weather: Extract -> Transform -> Load
        #    forecast_days=0 keeps Open-Meteo's forecast rows OUT of the
        #    observations table (see etl/extract_weather.py for why).
        df_weather_raw = fetch_weather_hourly(
            LAT, LON, TIMEZONE, past_days=7, forecast_days=0
        )
        assert_required_columns(
            df_weather_raw,
            ["ts", "temperature_c", "precipitation_mm", "windspeed_kmh"],
            "weather_raw",
        )
        df_weather_raw.to_csv(RAW_WEATHER_CSV, index=False)

        df_weather = clean_weather(df_weather_raw)
        assert_not_empty(df_weather, "weather_clean")
        weather_rows = len(df_weather)
        upsert_weather(DB_PATH, df_weather, location_id=1)

        # 2) Air Quality: Extract -> Transform -> Load
        df_aq_raw = fetch_air_quality_hourly(LAT, LON, TIMEZONE, past_days=7)
        assert_required_columns(df_aq_raw, ["ts", "pm25", "pm10", "no2", "o3"], "aq_raw")

        df_aq = clean_air_quality(df_aq_raw)
        assert_not_empty(df_aq, "aq_clean")
        aq_rows = len(df_aq)
        upsert_air_quality(DB_PATH, df_aq, location_id=1)

        # 3) Combined KPIs
        con = duckdb.connect(DB_PATH)
        with open(KPI_SQL, "r", encoding="utf-8") as f:
            sql = f.read()
        df_kpis = con.execute(sql).df()
        con.close()

        assert_not_empty(df_kpis, "kpi_output")
        kpi_rows = len(df_kpis)
        df_kpis.to_csv(KPI_OUT, index=False)

        # 4) Durable history — must run BEFORE prediction, since the model
        #    trains off the accumulated file rather than the rolling warehouse.
        append_daily_history(DB_PATH, location_id=1)

        # 5) Predict tomorrow + score every past forecast walk-forward
        prediction = predict_next_day(location_id=1)

        # 6) Charts + README (README now carries the forecast + accuracy block)
        make_charts(KPI_OUT, "reports/charts")
        make_prediction_charts("reports/charts")
        update_readme("README.md", KPI_OUT, prediction=prediction)

        append_run_log(weather_rows, aq_rows, kpi_rows, status="success")
        print("Pipeline complete (Weather + Air Quality + Charts + Forecast).")

    except Exception as e:
        append_run_log(weather_rows, aq_rows, kpi_rows, status="failed", message=str(e))
        raise


if __name__ == "__main__":
    main()

---
## 18. Automation — Daily ETL Workflow

`.github/workflows/daily_weather_etl.yml`

Runs at 11:10 UTC (~7:10 AM Toronto) and on manual dispatch.

The `git add README.md reports data/history` line is load-bearing: without `data/history` the observation record is regenerated and discarded on every run, and the model's training set never grows.


```yaml
name: Daily Weather ETL

on:
  schedule:
    # Runs daily at 11:10 UTC (~7:10 AM Toronto time)
    - cron: "10 11 * * *"
  workflow_dispatch:

permissions:
  contents: write

concurrency:
  group: daily-weather-etl
  cancel-in-progress: false

jobs:
  run-etl:
    runs-on: ubuntu-latest
    timeout-minutes: 15

    steps:
      - name: Checkout repository
        uses: actions/checkout@v4
        with:
          fetch-depth: 0

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: "pip"

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt

      - name: Run pipeline
        env:
          PYTHONPATH: ${{ github.workspace }}
        run: |
          python etl/run_weather_pipeline.py

      - name: Commit and push updates
        run: |
          git config user.name  "github-actions[bot]"
          git config user.email "github-actions[bot]@users.noreply.github.com"

          git pull --rebase --autostash origin main

          # Stage whole directories rather than individual files. The
          # prediction CSVs only appear once there is enough history to fit a
          # model, and `git add` fails the job if a named file doesn't exist.
          git add README.md reports data/history

          git diff --cached --quiet || git commit -m "Daily ETL update: KPIs, charts, and next-day forecast"

          git push origin main
```


---
## 19. Automation — CI Test Suite

`.github/workflows/tests.yml`

Runs pytest on every push and pull request.


```yaml
name: Tests

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]
  workflow_dispatch:

jobs:
  pytest:
    runs-on: ubuntu-latest
    timeout-minutes: 10

    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: "pip"

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt
          pip install -r requirements-dev.txt

      - name: Run test suite
        env:
          PYTHONPATH: ${{ github.workspace }}
        run: |
          pytest -q

      - name: Check the notebook is not stale
        env:
          PYTHONPATH: ${{ github.workspace }}
        run: |
          python tools/build_notebook.py --check
```


---
## 20. Test Suite

`pytest`, run in CI on every push.

The tests worth reading are the regression ones — each exists because the pipeline actually shipped the bug it guards against:

- a feature builder that would pair a February day with a June day across the history gap
- a model that published a 40.07 °C August forecast for Toronto


**`tests/test_transforms.py`** — Transform invariants


In [ ]:
# tests/test_transforms.py

"""Transform-layer invariants: bad data must not reach the warehouse."""

import pandas as pd

from etl.transform_weather import clean_weather
from etl.transform_air_quality import clean_air_quality


def test_impossible_temperatures_are_nulled():
    """A sensor spike of 999 C must not survive into the warehouse."""
    df = pd.DataFrame({
        "ts": pd.to_datetime(["2026-03-01 00:00", "2026-03-01 01:00",
                              "2026-03-01 02:00"]),
        "temperature_c":    [5.0, 999.0, -80.0],
        "precipitation_mm": [0.0, 0.0, 0.0],
        "windspeed_kmh":    [10.0, 10.0, 10.0],
    })

    out = clean_weather(df)

    assert out["temperature_c"].iloc[0] == 5.0
    assert pd.isna(out["temperature_c"].iloc[1]), "999 C should be nulled"
    assert pd.isna(out["temperature_c"].iloc[2]), "-80 C should be nulled"


def test_negative_precipitation_and_wind_are_nulled():
    """Negative rainfall is physically impossible."""
    df = pd.DataFrame({
        "ts": pd.to_datetime(["2026-03-01 00:00", "2026-03-01 01:00"]),
        "temperature_c":    [5.0, 5.0],
        "precipitation_mm": [-3.0, 1.0],
        "windspeed_kmh":    [-1.0, 8.0],
    })

    out = clean_weather(df)

    assert pd.isna(out["precipitation_mm"].iloc[0])
    assert pd.isna(out["windspeed_kmh"].iloc[0])
    assert out["precipitation_mm"].iloc[1] == 1.0


def test_duplicate_timestamps_are_dropped_and_rows_sorted():
    """Upserts key on ts, so duplicates must be resolved before loading."""
    df = pd.DataFrame({
        "ts": pd.to_datetime(["2026-03-01 02:00", "2026-03-01 00:00",
                              "2026-03-01 02:00"]),
        "temperature_c":    [7.0, 5.0, 7.0],
        "precipitation_mm": [0.0, 0.0, 0.0],
        "windspeed_kmh":    [10.0, 10.0, 10.0],
    })

    out = clean_weather(df)

    assert len(out) == 2, "duplicate timestamp should be dropped"
    assert out["ts"].is_monotonic_increasing, "rows must come out sorted"


def test_unparseable_timestamps_are_dropped():
    df = pd.DataFrame({
        "ts": ["2026-03-01 00:00", "not-a-date"],
        "temperature_c":    [5.0, 6.0],
        "precipitation_mm": [0.0, 0.0],
        "windspeed_kmh":    [10.0, 10.0],
    })

    out = clean_weather(df)

    assert len(out) == 1


def test_negative_pollutant_readings_are_nulled():
    """Concentrations cannot be below zero."""
    df = pd.DataFrame({
        "ts": pd.to_datetime(["2026-03-01 00:00", "2026-03-01 01:00"]),
        "pm25": [-5.0, 8.0],
        "pm10": [-2.0, 9.0],
        "no2":  [10.0, 11.0],
        "o3":   [40.0, 42.0],
    })

    out = clean_air_quality(df)

    assert pd.isna(out["pm25"].iloc[0])
    assert pd.isna(out["pm10"].iloc[0])
    assert out["pm25"].iloc[1] == 8.0

**`tests/test_features.py`** — Feature engineering — including the no-leakage-across-gaps property


In [ ]:
# tests/test_features.py

"""
Feature-engineering invariants.

The critical property here is NO LEAKAGE ACROSS CALENDAR GAPS. The real history
has a four-month hole in it; a lag feature that silently bridges that hole would
pair a February day with a June day and quietly poison the model.
"""

import pandas as pd

from etl.predict_temperature import (
    BASE_FEATURES,
    FEATURE_COLS,
    build_features,
    training_table,
)


def test_target_is_the_following_day(contiguous_daily):
    """target_temp_c on row d must equal avg_temp_c on row d+1."""
    feat = build_features(contiguous_daily)

    for i in range(len(feat) - 1):
        assert feat["target_temp_c"].iloc[i] == feat["avg_temp_c"].iloc[i + 1]
        assert feat["target_day"].iloc[i] == feat["day"].iloc[i] + pd.Timedelta(days=1)


def test_lag_features_use_only_past_and_present(contiguous_daily):
    """temp_lag1 is today, temp_lag2 is yesterday — never tomorrow."""
    feat = build_features(contiguous_daily)

    assert feat["temp_lag1"].iloc[5] == feat["avg_temp_c"].iloc[5]
    assert feat["temp_lag2"].iloc[5] == feat["avg_temp_c"].iloc[4]
    assert feat["temp_lag3"].iloc[5] == feat["avg_temp_c"].iloc[3]


def test_no_training_row_bridges_a_calendar_gap(gapped_daily):
    """
    The regression test that matters most.

    With a Feb block and a June block, no usable training row may pair a
    February feature with a June target. Every surviving row must sit inside
    one contiguous block.
    """
    train = training_table(build_features(gapped_daily))

    for _, row in train.iterrows():
        # target_day must be exactly one day after day — never a 100-day jump
        assert row["target_day"] - row["day"] == pd.Timedelta(days=1)
        # and the temperature must not teleport between seasons
        assert abs(row["target_temp_c"] - row["temp_lag1"]) < 10.0, (
            f"row on {row['day'].date()} bridges the gap: "
            f"temp_lag1={row['temp_lag1']} target={row['target_temp_c']}"
        )


def test_gap_reduces_usable_rows(gapped_daily, contiguous_daily):
    """20 gapped days must yield fewer training rows than 20 contiguous ones."""
    gapped_rows = len(training_table(build_features(gapped_daily)))
    contiguous_rows = len(training_table(build_features(contiguous_daily.head(20))))

    assert gapped_rows < contiguous_rows


def test_training_table_has_no_missing_values(contiguous_daily):
    """Anything with a NaN feature or target must be dropped before fitting."""
    train = training_table(build_features(contiguous_daily))

    assert not train[FEATURE_COLS + ["target_temp_c"]].isna().any().any()
    assert len(train) > 0


def test_rolling_window_needs_a_full_window(contiguous_daily):
    """temp_roll7 must be NaN until seven days are available."""
    feat = build_features(contiguous_daily)

    assert pd.isna(feat["temp_roll7"].iloc[5]), "day 6 cannot have a 7-day mean"
    assert not pd.isna(feat["temp_roll7"].iloc[6]), "day 7 should have one"


def test_seasonal_columns_exist_but_are_not_in_base_features():
    """The seasonal pair is built every run but only used when it's earned."""
    assert "doy_sin" in FEATURE_COLS
    assert "doy_sin" not in BASE_FEATURES

**`tests/test_prediction_guards.py`** — Regression tests for both extrapolation guards


In [ ]:
# tests/test_prediction_guards.py

"""
Regression tests for the two extrapolation guards.

Both of these exist because of a real defect: on 2026-08-25 the pipeline
published a next-day forecast of 40.07 C for Toronto in August. The cause was
seasonal day-of-year features fitted on 21 rows covering only February and
June, then asked to extrapolate into late August.
"""

import pandas as pd
import pytest

from etl.predict_temperature import (
    BASE_FEATURES,
    MAX_DAILY_SWING_C,
    SEASONAL_FEATURES,
    apply_sanity_bounds,
    build_features,
    select_features,
    training_table,
)


def _train_from(daily):
    return training_table(build_features(daily))


# ---------------------------------------------------------------------------
# Guard 1 — seasonal features are gated on real annual coverage
# ---------------------------------------------------------------------------
def test_seasonal_features_off_with_short_history(contiguous_daily):
    """40 days spanning 40 days is nowhere near a year."""
    cols = select_features(_train_from(contiguous_daily))

    assert cols == BASE_FEATURES
    assert "doy_sin" not in cols


def test_seasonal_features_off_with_gappy_partial_year(gapped_daily):
    """
    The exact failure case: February plus June, wide span, very few rows.
    Span alone must not be enough to unlock the seasonal features.
    """
    cols = select_features(_train_from(gapped_daily))

    assert "doy_sin" not in cols, "sparse coverage must not enable seasonality"


def test_seasonal_features_on_with_a_full_year():
    """Once there is a genuine year of daily history, seasonality is allowed."""
    days = pd.date_range("2025-01-01", periods=420, freq="D")
    daily = pd.DataFrame({
        "day": days,
        "location_id": 1,
        "avg_temp_c": [10.0 + 15.0 * pd.Timestamp(d).dayofyear / 365.0
                       for d in days],
        "max_temp_c": [15.0] * len(days),
        "total_precip_mm": [0.0] * len(days),
        "avg_windspeed_kmh": [10.0] * len(days),
        "hours_observed": 24,
    })

    cols = select_features(_train_from(daily))

    assert "doy_sin" in cols and "doy_cos" in cols
    assert cols == BASE_FEATURES + SEASONAL_FEATURES


# ---------------------------------------------------------------------------
# Guard 2 — physical sanity bounds on the published prediction
# ---------------------------------------------------------------------------
def test_the_40_degree_bug_is_clamped(contiguous_daily):
    """The literal regression test: 40 C must never be published again."""
    train = _train_from(contiguous_daily)
    baseline = 17.44

    bounded, was_clamped = apply_sanity_bounds(40.07, baseline, train)

    assert was_clamped is True
    assert bounded < 40.07
    assert bounded <= baseline + MAX_DAILY_SWING_C


def test_absurd_cold_is_clamped(contiguous_daily):
    train = _train_from(contiguous_daily)
    baseline = 17.44

    bounded, was_clamped = apply_sanity_bounds(-30.0, baseline, train)

    assert was_clamped is True
    assert bounded >= baseline - MAX_DAILY_SWING_C


def test_plausible_prediction_passes_through_untouched(contiguous_daily):
    """A sane forecast must not be altered — the guard is a backstop, not a filter."""
    train = _train_from(contiguous_daily)

    bounded, was_clamped = apply_sanity_bounds(18.67, 17.44, train)

    assert was_clamped is False
    assert bounded == pytest.approx(18.67)


def test_clamp_never_moves_more_than_the_swing_limit(contiguous_daily):
    """Whatever the model emits, the result stays within the swing band."""
    train = _train_from(contiguous_daily)
    baseline = 12.0

    for raw in (-500.0, -20.0, 0.0, 12.0, 25.0, 500.0):
        bounded, _ = apply_sanity_bounds(raw, baseline, train)
        assert baseline - MAX_DAILY_SWING_C <= bounded <= baseline + MAX_DAILY_SWING_C

**`tests/test_history_and_quality.py`** — History upsert semantics and the data-quality gate


In [ ]:
# tests/test_history_and_quality.py

"""
History-store upsert semantics and the data-quality gate.

The history file is the only durable copy of the observation record — the
warehouse holds just Open-Meteo's rolling window. If an upsert ever dropped
older days, the training set would silently shrink.
"""

import os

import pandas as pd
import pytest

from etl import history_store
from etl.data_quality import assert_not_empty, assert_required_columns


# ---------------------------------------------------------------------------
# History store
# ---------------------------------------------------------------------------
@pytest.fixture
def temp_history(tmp_path, monkeypatch):
    """Point HISTORY_CSV at a throwaway file so tests never touch the repo."""
    path = tmp_path / "history" / "daily_observations.csv"
    monkeypatch.setattr(history_store, "HISTORY_CSV", str(path))
    return str(path)


def _write_history(path, days, temps):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    pd.DataFrame({
        "day": days,
        "location_id": 1,
        "avg_temp_c": temps,
        "max_temp_c": [t + 4 for t in temps],
        "total_precip_mm": [0.0] * len(days),
        "avg_windspeed_kmh": [10.0] * len(days),
        "hours_observed": [24] * len(days),
    }).to_csv(path, index=False)


def test_load_returns_empty_when_no_history_yet(temp_history):
    out = history_store.load_daily_history(1)
    assert out.empty


def test_load_round_trips_and_parses_dates(temp_history):
    _write_history(temp_history, ["2026-03-01", "2026-03-02"], [5.0, 6.0])

    out = history_store.load_daily_history(1)

    assert len(out) == 2
    assert pd.api.types.is_datetime64_any_dtype(out["day"])
    assert out["day"].is_monotonic_increasing


def test_load_filters_by_location(temp_history):
    os.makedirs(os.path.dirname(temp_history), exist_ok=True)
    pd.DataFrame({
        "day": ["2026-03-01", "2026-03-01"],
        "location_id": [1, 2],
        "avg_temp_c": [5.0, 25.0],
        "max_temp_c": [9.0, 30.0],
        "total_precip_mm": [0.0, 0.0],
        "avg_windspeed_kmh": [10.0, 10.0],
        "hours_observed": [24, 24],
    }).to_csv(temp_history, index=False)

    out = history_store.load_daily_history(1)

    assert len(out) == 1
    assert out["avg_temp_c"].iloc[0] == 5.0


def test_history_columns_are_stable(temp_history):
    """Downstream feature code depends on these names."""
    _write_history(temp_history, ["2026-03-01"], [5.0])

    out = history_store.load_daily_history(1)

    for col in ["day", "avg_temp_c", "total_precip_mm", "avg_windspeed_kmh"]:
        assert col in out.columns


# ---------------------------------------------------------------------------
# Data quality gate
# ---------------------------------------------------------------------------
def test_missing_columns_raise():
    df = pd.DataFrame({"ts": [1], "temperature_c": [2.0]})

    with pytest.raises(ValueError, match="missing required columns"):
        assert_required_columns(df, ["ts", "temperature_c", "windspeed_kmh"], "weather")


def test_present_columns_pass():
    df = pd.DataFrame({"ts": [1], "temperature_c": [2.0]})

    assert_required_columns(df, ["ts", "temperature_c"], "weather")  # must not raise


def test_empty_frame_raises():
    with pytest.raises(ValueError, match="empty"):
        assert_not_empty(pd.DataFrame(), "weather_clean")


def test_none_frame_raises():
    with pytest.raises(ValueError):
        assert_not_empty(None, "weather_clean")


def test_populated_frame_passes():
    assert_not_empty(pd.DataFrame({"a": [1]}), "weather_clean")  # must not raise

---
## Pipeline Summary

| Stage | File | What it does |
|---|---|---|
| Config | `etl/config.py` | City coordinates and file paths |
| Extract | `etl/extract_weather.py` | Hourly weather, observations only |
| Extract | `etl/extract_openaq.py` | Hourly air quality |
| Transform | `etl/transform_weather.py` | Cleans and validates weather |
| Transform | `etl/transform_air_quality.py` | Cleans and validates air quality |
| Load | `etl/load_weather_duckdb.py` | Idempotent upsert into DuckDB |
| Load | `etl/load_air_quality_duckdb.py` | Idempotent upsert into DuckDB |
| History | `etl/history_store.py` | Durable append-only observation record |
| Predict | `etl/predict_temperature.py` | Next-day forecast + walk-forward scoring |
| Report | `etl/make_charts_combined.py` | 30-day KPI charts |
| Report | `etl/make_prediction_chart.py` | Forecast vs actual, error per day |
| Report | `etl/update_readme_weather.py` | Rewrites the README |
| Quality | `etl/data_quality.py` | Validation gate and run log |
| Orchestrate | `etl/run_weather_pipeline.py` | Entry point |

---

*Built by Andre Felix — data updated daily via GitHub Actions*
